# Week 10 - Day 3 Lab
# AI Agent Frameworks with LangChain & LangGraph

---

## Learning Objectives

By the end of this lab, you will be able to:

- Build an AI Agent using LangChain.
- Register and use custom tools.
- Understand how LangChain simplifies the manual agent loop.
- Build a simple workflow using LangGraph.
- Add memory to an AI Agent.
- Extend the agent with your own custom tool.

Estimated Time: 2.5 Hours

# Step 1: Install Required Libraries

Run the following cell to install all required libraries.

In [1]:
!pip -q install -U langchain
!pip -q install -U langgraph
!pip -q install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


# Step 2: Configure Gemini

We will use Google's Gemini model throughout this lab.

In [7]:
import os
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


## Create the Gemini LLM

Import the required class and initialize the model.

💡 Hint

The class name starts with

ChatGoogle...

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0
)

### Test Your Model

If everything is configured correctly, the model should respond.

Expected Output

Hello! How can I help you today?

In [10]:
response = llm.invoke("Say Hello!")

print(response.text)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Hello! How can I help you today?


# Creating Your First Tool

A tool is simply a Python function decorated with `@tool`.

Today we'll build a Weather Tool.

## Task

Complete the dictionary below.

💡 Hint

The dictionary key should be the city name.

The value should be the weather.

In [11]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    Returns the weather of a city.
    """

    weather = {
        "Lahore": "Sunny, 34°C",
        "Karachi": "Humid, 31°C",
        "Islamabad": "Cloudy, 28°C"
    }

    return weather.get(city, "Weather unavailable")

## Test Your Tool

In [12]:
print(get_weather.invoke({"city": "Lahore"}))
print(get_weather.invoke({"city": "Karachi"}))
print(get_weather.invoke({"city": "Islamabad"}))
print(get_weather.invoke({"city": "London"}))

Sunny, 34°C
Humid, 31°C
Cloudy, 28°C
Weather unavailable


## Challenge

Add TWO more cities.

Suggested cities:

- Peshawar
- Quetta

# Calculator Tool

Let's create another tool.

## Task

Complete the function.

💡 Hint

Python has a function that evaluates mathematical expressions.

In [13]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    Returns the weather of a city.
    """

    weather = {
        "Lahore": "Sunny, 34°C",
        "Karachi": "Humid, 31°C",
        "Islamabad": "Cloudy, 28°C",
        "Peshawar": "Sunny, 35°C",
        "Quetta": "Clear, 26°C"
    }

    return weather.get(city, "Weather unavailable")

In [14]:
from langchain.tools import tool

@tool
def calculator(expression: str):

    """
    Evaluate a mathematical expression.
    """

    return eval(expression)

## Test

Try

25*18

Expected Output

450

In [15]:
print(calculator.invoke({"expression": "25*18"}))

450


# Registering Tools

The AI Agent needs to know which tools it can use.

## Task

Complete the list below.

💡 Hint

Use the function names only.

In [16]:
tools = [

    get_weather,

    calculator

]

# Binding Tools to Gemini

Gemini must know which tools are available.

## Task

Complete the following line.

💡 Hint

The method starts with

bind...

In [17]:
llm_with_tools = llm.bind_tools(tools)

## Test Tool Calling

Ask Gemini about the weather.

Observe the response carefully.

Does Gemini answer directly?

Or

Does it request a tool?

In [19]:
response = llm_with_tools.invoke(

    "What's the weather in Lahore?"

)

response

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "Lahore"}'}, '__gemini_function_call_thought_signatures__': {'call_92536': 'El4KXAERTTIPJs7MhmiRImu+DwZe309bYFx8av8AUKtPtoN9mx8coz2jVnzcliadDp+osUBeLMcIgdXsz8J+LfdJO47nqmgtUyqJiuiXjFigyK2WGOD45Xz/6KzY+PEU'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a08124-c52c-7fb0-9782-3b24a03c001b-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Lahore'}, 'id': 'call_92536', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 17, 'total_tokens': 110, 'input_token_details': {'cache_read': 0}})

# Tool Registry

When Gemini requests a tool, Python needs to know which function to execute.

## Task

Complete the registry.

💡 Hint

The key is a string.

The value is the function.

In [20]:
tool_registry = {

    "get_weather": get_weather,

    "calculator": calculator

}

# Executing Tool Calls

Complete the missing lines.

💡 Hint

1. Find the tool.

2. Execute it.

3. Print the result.

In [21]:
for tool_call in response.tool_calls:

    tool_name = tool_call["name"]

    args = tool_call["args"]

    # Find the tool
    tool = tool_registry[tool_name]

    # Execute the tool
    result = tool.invoke(args)

    print(result)

Sunny, 34°C


# Creating a LangChain Agent

Earlier we manually:

- Parsed tool calls
- Executed tools
- Returned observations

LangChain automates all of that.

Complete the missing imports.

💡 Hint

The required classes are:

- AgentExecutor
- create_tool_calling_agent

In [25]:
!pip install -q \
    "langchain==0.3.27" \
    "langchain-core==0.3.72" \
    "langchain-google-genai"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.8/442.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.6 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.
langgraph-sdk 0.4.3 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.72 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.72 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.72 which is incompatible.


In [26]:
import langchain
import langchain_core

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)

LangChain: 1.4.0
LangChain Core: 1.6.2


## Build the Agent

Fill in the blanks.

Don't worry if you don't remember every function—we discussed this in today's lecture.

💡 Hint

Use:

- llm
- tools
- prompt

In [30]:
!pip install -q -U langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.27 requires langchain-core<1.0.0,>=0.3.72, but you have langchain-core 1.6.2 which is incompatible.
langchain 0.3.27 requires langchain-text-splitters<1.0.0,>=0.3.9, but you have langchain-text-splitters 1.1.2 which is incompatible.
langchain-google-genai 2.1.9 requires langchain-core<0.4.0,>=0.3.68, but you have langchain-core 1.6.2 which is incompatible.


In [32]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

In [34]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use tools when needed."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

print("Prompt created successfully!")

Prompt created successfully!


In [35]:
agent = create_tool_calling_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

## Test the Agent

In [36]:
result = agent_executor.invoke({

    "input":"Should I carry an umbrella in Islamabad?"

})

print(result["output"])



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_weather` with `{'city': 'Islamabad'}`


Cloudy, 28°C

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "It is currently cloudy and 28°C in Islamabad. While it's not actively raining, the cloudy conditions mean there's a chance of precipitation, so you might want to carry an umbrella just in case!", 'index': 0, 'extras': {'signature': 'El4KXAERTTIPg7VKELiAA7p1sLmRpBAd7lNV9VUClITg+IIiWZ/RHRr7pTcmyvRm8LtWfV/jKiWSC6yjQ5tmEvW64/j5+iuSQ2svzDPS0FszVtnro0A4Wil/qMhCCZO3'}}]

> Finished chain.
[{'type': 'text', 'text': "It is currently cloudy and 28°C in Islamabad. While it's not actively raining, the cloudy conditions mean there's a chance of precipitation, so you might want to carry an umbrella just in case!", 'index': 0, 'extras': {'signature': 'El4KXAERTTIPg7VKELiAA7p1sLmRpBAd7lNV9VUClITg+IIiWZ/RHRr7pTcmyvRm8LtWfV/jKiWSC6yjQ5tmEvW64/j5+iuSQ2svzDPS0FszVtnro0A4Wil/qMhCCZO3'}}]


# LangGraph State

LangGraph stores information inside a shared State object.

## Task

Complete the state.

💡 Hint

Today's slides showed three important fields.

One of them is

messages

In [37]:
from typing import TypedDict

class AgentState(TypedDict):

    messages: list

    current_agent: str

    next: str

# Creating Nodes

Every node is simply a Python function.

Complete the chatbot node.

💡 Hint

The node should

1. Read the state

2. Call the LLM

3. Return the updated state

In [38]:
def chatbot_node(state):

    response = llm.invoke(state["messages"])

    return {"messages": state["messages"] + [response]}

# Memory

Let's build a very small conversation memory.

Complete the class.

💡 Hint

Remember:

append()

returns nothing.

In [39]:
class ConversationMemory:

    def __init__(self):
        self.messages = []

    def add(self, message):
        self.messages.append(message)

    def get_context(self):
        return self.messages

## Challenge

Modify your memory so it only stores the latest **5** messages.

Hint:

Python list slicing may be useful.

# Smart Travel Assistant

Congratulations!

You now know how to build AI Agents using LangChain.

## Your Task

Build a Travel Assistant.

Requirements

✅ Weather Tool

✅ Calculator Tool

Create ONE new tool.

Choose ONE:

- Restaurant Tool
- Movie Tool
- Hotel Tool
- Currency Converter

The assistant should answer questions like:

"I'm visiting Lahore tomorrow.

What's the weather?

Recommend a restaurant.

How much will dinner cost for 4 people if each meal costs $25?"

---

## Bonus Challenge

Can your assistant remember the user's city without asking again?

Example

User:

"I'm visiting Lahore."

Later...

"What's the weather tomorrow?"

The assistant should understand that the city is still Lahore.

# **1. Memory — keep only the latest 5 messages**

In [40]:
class ConversationMemory:

    def __init__(self):
        self.messages = []

    def add(self, message):
        self.messages.append(message)

        # Keep only the latest 5 messages
        self.messages = self.messages[-5:]

    def get_context(self):
        return self.messages

# **2. Smart Travel Assistant**

In [41]:
from langchain_core.tools import tool
import re


@tool
def get_weather(city: str, date: str = "today") -> str:
    """
    Get the weather for a city.
    """

    weather = {
        "lahore": "Sunny, 34°C",
        "karachi": "Humid, 31°C",
        "islamabad": "Cloudy, 28°C",
        "peshawar": "Sunny, 35°C",
        "quetta": "Clear, 26°C"
    }

    return weather.get(city.lower(), "Weather information unavailable.")


@tool
def calculator(expression: str) -> str:
    """
    Calculate a mathematical expression.
    """

    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return "Invalid mathematical expression."

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"


@tool
def recommend_restaurant(city: str) -> str:
    """
    Recommend a restaurant in a city.
    """

    restaurants = {
        "lahore": "Haveli Restaurant",
        "karachi": "Kolachi Restaurant",
        "islamabad": "Monal Restaurant",
        "peshawar": "Nisar Charsi Tikka",
        "quetta": "Lehri Sajji House"
    }

    return restaurants.get(
        city.lower(),
        "No restaurant recommendation available."
    )

# **3. Create the prompt with memory**

In [42]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a helpful travel assistant.

        You can:
        - check weather
        - recommend restaurants
        - perform calculations

        Use the tools whenever appropriate.

        Remember information from the conversation,
        such as the user's destination city.
        """
    ),

    MessagesPlaceholder(
        variable_name="chat_history",
        optional=True
    ),

    ("human", "{input}"),

    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    )
])

# **4. Build the LangChain agent**

In [43]:
from langchain_classic.agents import (
    AgentExecutor,
    create_tool_calling_agent
)

agent = create_tool_calling_agent(
    llm,
    tools,
    prompt
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# **5. Add conversation memory**

In [44]:
from langchain_core.messages import HumanMessage, AIMessage

memory = ConversationMemory()

In [45]:
def chat(user_input):

    response = agent_executor.invoke({
        "input": user_input,
        "chat_history": memory.get_context()
    })

    answer = response["output"]

    # Save user message
    memory.add(
        HumanMessage(content=user_input)
    )

    # Save assistant response
    memory.add(
        AIMessage(content=answer)
    )

    return answer

# **6. Test the Travel Assistant**

In [46]:
print(
    chat(
        """
        I'm visiting Lahore tomorrow.

        What's the weather?

        Recommend a restaurant.

        How much will dinner cost for 4 people
        if each meal costs $25?
        """
    )
)



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_weather` with `{'city': 'Lahore'}`


Sunny, 34°C
Invoking: `calculator` with `{'expression': '4 * 25'}`


100

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Here is the information for your trip to Lahore:\n\n* **Weather:** It will be sunny and 34°C tomorrow, so stay hydrated and wear light clothing!\n* **Restaurant Recommendation:** I recommend checking out **Monal Lahore**, located in Liberty Gulberg, or **Andaaz Restaurant** in the historic Walled City for a fantastic rooftop dining experience with traditional Pakistani cuisine.\n* **Dinner Cost:** For 4 people at $25 per meal, the total cost for dinner will be **$100**.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIP1tzsDk43obhG2kgmLTFxPeEhO+A7d64zQAdPopXKPhB+rzkbNWCN0JQPxG9uEbXYF4SMPiGQEOIozWLDyB1Ye6N7nT9mu4uieFpT/i0OuXbL4JkxkSui'}}]

> Finished chain.
[{'type': 'text', 'text': 'Here is the information for your trip to Lahore:\n\n* **Weather:** It will be sunny and 34°C tomorrow, so stay hydrated and wear light clothing!\n* **Restaurant Recommendation:** I recommend checking out **Monal Lahore**, located in Liberty Gulberg, or **Andaaz Restaurant** in th

In [47]:
print(chat("I'm visiting Lahore."))



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "I remember you mentioned you're visiting Lahore tomorrow! \n\nIs there anything else you'd like to know about your trip, such as more restaurant options, things to do, or the weather forecast?", 'index': 0, 'extras': {'signature': 'El4KXAERTTIPLtWOVd7/U6hG/ztX3mmTlquWemtkz/w2feIXgDFAIRv4NStJpZ+wQDbEAC5xH98F6p3UbrPePSgEhHYolo7uI4OvS/h1Xl7D/7uFeKzDDc+u9RJkhvmj'}}]

> Finished chain.
[{'type': 'text', 'text': "I remember you mentioned you're visiting Lahore tomorrow! \n\nIs there anything else you'd like to know about your trip, such as more restaurant options, things to do, or the weather forecast?", 'index': 0, 'extras': {'signature': 'El4KXAERTTIPLtWOVd7/U6hG/ztX3mmTlquWemtkz/w2feIXgDFAIRv4NStJpZ+wQDbEAC5xH98F6p3UbrPePSgEhHYolo7uI4OvS/h1Xl7D/7uFeKzDDc+u9RJkhvmj'}}]


In [48]:
print(chat("What's the weather tomorrow?"))



> Entering new AgentExecutor chain...


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Invoking: `get_weather` with `{'city': 'Lahore'}`


Sunny, 34°C

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The weather in Lahore tomorrow will be sunny with a temperature of 34°C.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPvjmuxaqrAz+b8X3KY+RruT5qFYDvQeenocM6KDlIU4uEEumLQ0BcTveA0Tj3EJsSgwvlF9fxM/6K3K8sopVZs5pQPPMAZxYK4+5wpqa8dCyEoumXLJ2w'}}]

> Finished chain.
[{'type': 'text', 'text': 'The weather in Lahore tomorrow will be sunny with a temperature of 34°C.', 'index': 0, 'extras': {'signature': 'El4KXAERTTIPvjmuxaqrAz+b8X3KY+RruT5qFYDvQeenocM6KDlIU4uEEumLQ0BcTveA0Tj3EJsSgwvlF9fxM/6K3K8sopVZs5pQPPMAZxYK4+5wpqa8dCyEoumXLJ2w'}}]
